# Behavioral tests of the ranked value algorithm

The thesis tests a **ranked value algorithm**: the animal ranks the three options (best, second, third)
and its choices are gated by comparisons between adjacent ranks. Mice switch toward the alternative they
believe is better, beyond win-stay/lose-shift (Habit), but a softmax value learner predicts that too.
These five tests look for signatures a **softmax value learner (our kernel + forgetting Q-learner, fitted
per mouse) does not produce** and a rank algorithm does:

1. *Stay depends on the second best, not the third.* Rank: the best alternative lowers staying, the worst
   does not (`w_best < 0`, `w_worst` about 0). Softmax: both lower it, the worst less.
2. *Abrupt switching at believed rank crossings.* Rank: P(leave) is a step at `D = 0` (`D` = own value
   minus the best alternative's), and the mouse adopts the new best at once. Softmax: graded.
3. *The third option is treated categorically.* Rank: P(choose the believed worst) is flat in its gap to
   the second best. Softmax: it falls with the gap.
4. *Ranks matter more than values.* Rank: held-out fits prefer rank indicators to values, and staying at a
   rank does not depend on the payout's magnitude. Softmax: the reverse.
5. *Transfer and inference.* Rank: performance improves across problems (the structure transfers), and
   after a block change the first leave goes to the old zero arm (it now holds the zeroed arm's value).
   Softmax: no transfer across problems, and the leave goes to the third arm, whose last payout is higher.

**How every readout is read.** Each mouse's readout is compared with 100 replays of each agent fitted to
*that mouse* (WSLS (Magnitude), WSLS (Habit), and the kernel + forgetting Q-learner), played on the
mouse's own problems and analyzed identically, so the schedule and the analysis are controlled for. The
Q-learner is the key comparator: a rank signature means the mice differ from it in the predicted
direction. Calibrated z = (observed - replay mean) / replay sd, with a two-sided empirical p.

Section 1 runs every test on simulated agents where the answer is known; section 2 selects the real
data; section 3 reports the baseline numbers; section 4 runs the tests on the mice.

In [ ]:
# ruff: noqa: E402  (the repository root is put on sys.path first)
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/behavior_import").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.agents import null_agents
from src.behavior_analysis.data import drifting_task_rules as task_rules
from src.behavior_analysis.data import problem_selection
from src.behavior_analysis.rank_tests import rank_state
from src.behavior_analysis.strategy_models import strategy_models
from src.behavior_analysis.rank_tests import rank_signatures as signatures
from src.behavior_visualization.rank_tests import plot_rank_signatures as rank_plots
from src.pipelines.choice_logit import choice_logit_values as logit_pipe
from src.pipelines.strategy_models import null_agents as nulls
from src.pipelines.rank_tests import rank_baseline
from src.pipelines.rank_tests import rank_signatures as pipe
from src.pipelines.switching import switch_regression as switch
from src.settings import get_unchecked_settings
from tests.rank_fixtures import inferring_rank_mouse, rank_mouse, softmax_mouse
from tests.switch_regression_fixtures import table_from_rows


def show(base):
    # figures are saved (pdf, png, svg, caption) by the plot functions; display the png here
    display(Image(filename=f"{base}.png"))

In [ ]:
# ---- parameters (edit here) ------------------------------------------------------------
COHORTS = [1, 3]  # cohorts whose drifting problems are pooled in the real-data section
DATA_ROOT = None  # None: BANDIT_DATA_ROOT. Loading is cache-first and never writes to the share.
OUTPUT_DIR = REPO_ROOT / "results" / "figures" / "rank-tests"

# The last drifting problem kept per cohort (None: all). Every cohort of COHORTS needs an entry.
# Cohort 1 is cut after drifting problem 7: two mice end there, and the three that continue
# return from a four-week surgery / miniscope break with degraded behavior (problem 8 onward,
# from 2026-04-28). Meg's "problem 11" counts the four blocked problems before the drifting
# ones: it is drifting problem 7.
# Changing the cutoff to 11 is one edit here (the "problem 11" counting would then be 15).
MAX_PROBLEM_BY_COHORT = {1: 7, 3: None}

REPLICATES = pipe.DEFAULT_REPLICATES  # replays per mouse and agent
JOBS = 2  # parallel workers
STRATEGY_STARTS = strategy_models.Q_STARTS
SEED = 20261111
CACHE_ROOT = get_unchecked_settings().cache_dir
CACHE_DIR = CACHE_ROOT / "rank_tests"  # disposable
SHARED_STRATEGY_CACHE = CACHE_ROOT / "strategy_tests"  # the strategy comparison's cache, read only
SHARED_SWITCH_CACHE = CACHE_ROOT / "switch_regression"  # the switch regression's cache
SWITCH_SEED = 20261109  # the seed of the switch regression's calibration in the earlier notebook
SWITCH_REPLICATES = switch.DEFAULT_REPLICATES
BASELINE_MODELS = (
    strategy_models.WSLS_MAGNITUDE,
    strategy_models.WSLS_HABIT,
    strategy_models.Q_LEARNER,
    strategy_models.KERNEL_Q,
)

# Section 1: simulated agents. Each type is simulated for SYNTHETIC_MICE mice.
SYNTHETIC = dict(n_problems=10, sessions=4, trials=200)
SYNTHETIC_MICE = 3

## 1. Simulated agents with a known answer

Three simulated agents play the drifting task on layouts like the mice's, and every test runs on each.
**Softmax** is a value learner with learning rate 1 (its value is the last payout, the believed value)
and a softmax choice rule. **Rank** is a toy agent that sits on the arm it believes best and moves to the
best alternative the moment its payout falls below that alternative's believed value; the arm it believes
worst is never compared with. **Rank + block inference** also goes to the arm it believes worth 0 when its
arm pays 0 (the old zero arm after a block change). **Learner** is a hand-built mouse whose P(best)
rises across problems, for the transfer test. The table shows each readout of the tests for each agent;
`predicted` is the direction in which a rank algorithm lies above (+1) or below (-1) a softmax learner,
and the last column checks it where the agents can differ.

In [ ]:
def synthetic_cohort(make, label, seed0):
    return pd.concat(
        [make(seed=seed0 + k, mouse=f"{label}-{k}", **SYNTHETIC) for k in range(SYNTHETIC_MICE)],
        ignore_index=True,
    )


def learning_mice(label, seed0):
    """Problems with best arm A (4 pulses, B 2, C 0), chosen more often as problems go by."""
    ports, mags = ("A1", "B3", "C2"), (4, 2, 0)
    frames = []
    for k in range(SYNTHETIC_MICE):
        rng = np.random.default_rng(seed0 + k)
        rows = []
        for problem, p_best in enumerate(np.linspace(0.3, 0.95, SYNTHETIC["n_problems"]), start=1):
            for t in range(SYNTHETIC["sessions"] * SYNTHETIC["trials"]):
                arm = 0 if rng.random() < p_best else int(rng.integers(1, 3))
                rows.append(
                    (
                        (1, problem),
                        0,
                        t,
                        False,
                        ports[arm],
                        mags[arm],
                        dict(zip(ports, mags, strict=True)),
                    )
                )
        frames.append(table_from_rows(rows, mouse=f"{label}-{k}"))
    return pd.concat(frames, ignore_index=True)


agents = {
    "Softmax": synthetic_cohort(softmax_mouse, "softmax", 100),
    "Rank": synthetic_cohort(rank_mouse, "rank", 200),
    "Rank + Block Inference": synthetic_cohort(inferring_rank_mouse, "infer", 300),
    "Learner": learning_mice("learner", 400),
}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    synthetic = {label: pipe.analyze_mice(table, n_jobs=JOBS) for label, table in agents.items()}
means = pd.DataFrame(
    {label: s.readouts.drop(columns="mouse").mean() for label, s in synthetic.items()}
)
means["predicted"] = [signatures.PREDICTED.get(r, np.nan) for r in means.index]
means["test"] = [signatures.which_test(r) for r in means.index]
means["rank - softmax"] = means["Rank"] - means["Softmax"]
means["as predicted"] = np.where(
    means["predicted"].isna(), "", np.sign(means["rank - softmax"]) == means["predicted"]
)
with pd.option_context("display.max_rows", None, "display.width", 200, "display.max_columns", None):
    display(means[means["test"].notna()].sort_values(["test", "predicted"]).round(3))

**Reading the table.** Tests 1 to 4: the Rank agent lies above Softmax in the predicted direction for every readout the toy agents can separate: `w_worst` (-0.04 against -0.24), `w_diff` (1.13 against 0.76), `step_weight` (5.9 against 0.0), `cross_leave_1` and `cross_adopt_1` (0.99 and 0.97 against 0.83 and 0.70), `worst_slope` (+0.14 against -0.81), `scale_slope_first` (0.07 against 0.65; smaller is rank-like), `rank_minus_value` and `sw_rank_minus_value` (positive for Rank, negative for Softmax). `cross_t50` is 1 for both: the softmax learner also adopts the new best within a trial on average, so it cannot separate them here (and on the mice it is censored at 11 for every mouse, see section 4). Test 5: the toy agents do not learn across problems, so `slope_p100` is the same for Rank and Softmax (0.006); the Learner built to improve has `slope_p100` 0.073 and `slope_threshold` -18, so the readouts see a transfer that exists. Only the block-inferring agent goes to the old zero arm (`block_inherit` 0.92, against 0.35 for Rank and 0.26 for Softmax). `as predicted` is blank where the toy agents have no predicted difference (the Learner and block agents are built for tests 5a and 5b only).

## 2. Data selection

**Cohort 1: drifting problems 1 to 7 only.** After a four-week surgery / miniscope break only MY_71_L,
MY_71_R and MY_72_L continue (problem 8 onward, from 2026-04-28), with degraded behavior; MY_72_N and
MY_72_R end at drifting problem 7. Meg's "problem 11" counts the four blocked (non-drifting) problems
before the drifting ones. **Cohort 3: all drifting problems.** The trial table already holds drifting
problems only, for both cohorts: the loader (`load_drifting_problems` through `drifting_problems`) keeps
the drifting problems and numbers them within the cohort, leaving the blocked ones out. The cutoff is
`MAX_PROBLEM_BY_COHORT`; the per-mouse fits (kernel Q, WSLS (Habit), WSLS (Magnitude), Q-learner) are
refitted on the selected table (every cache key contains the mouse's trials, so a mouse whose data
changed is never served a stale fit).

In [ ]:
trials = logit_pipe.load_drifting_trial_table(COHORTS, DATA_ROOT)
selected = problem_selection.select_problems(trials, MAX_PROBLEM_BY_COHORT)
print("trials before / after the selection:", len(trials), "/", len(selected))
print("problems dropped per mouse:")
display(problem_selection.dropped_table(trials, selected))
counts = selected.groupby(["mouse"]).agg(
    n_trials=("trial", "size"), n_problems=("problem", "nunique")
)
display(counts)

In [ ]:
# What the schedules offer: the magnitude sets and what a block change does.
state_by_mouse = {m: rank_state.mouse_state(rows) for m, rows in selected.groupby("mouse")}
offered = pd.concat([pd.DataFrame(s.true).assign(mouse=m) for m, s in state_by_mouse.items()])
sets = offered[[0, 1, 2]].apply(lambda r: tuple(sorted(r.astype(int))), axis=1).value_counts()
print("magnitude sets offered (sorted triples), share of trials:")
display((sets / sets.sum()).round(3).rename("share"))
print("arms at 0 per trial: always", sorted(set((offered[[0, 1, 2]] == 0).sum(axis=1))))
print(
    "highest magnitude on a trial:",
    offered[[0, 1, 2]].max(axis=1).value_counts(normalize=True).sort_index().round(3).to_dict(),
)
changes = task_rules.block_changes(selected)
print(f"block changes: {len(changes)}; the zeroed arm was the mouse's most chosen:")
display(changes["zeroed_choice_rank"].value_counts())
print(
    "the old zero arm took the zeroed arm's value in",
    int(changes["inherits_value"].sum()),
    "of",
    len(changes),
)

**Schedule structure** (cohorts 1 and 3 after the selection). Exactly one arm pays 0 on every trial, the other two
pay distinct values from 1 to 4, so the magnitude sets are `{0, a, b}` with `a`, `b` in 1..4: the absolute
scale varies across blocks and problems (the same rank occurs at different magnitudes), so the scale test
(4b) is testable. At a block change the task zeroes the non-zero arm the mouse chose most in the block and
gives its value to the arm that was at zero, in every change (the printed counts), so the new best can be
inferred without sampling it: test 5b is testable.

## 3. Baseline check on the selected data

The headline numbers of the earlier analyses on the selected table next to the unselected one: the switch
regression's value weights calibrated against the four null agents (WSLS, WSLS (Recency), WSLS (Graded),
WSLS (Habit) as before; here WSLS (Magnitude) is the first), and the held-out nats per trial of WSLS
(Magnitude), WSLS (Habit), the Q-learner and the kernel Q-learner. Finished mice are read from the shared
caches; only the mice whose data changed are refitted.

In [ ]:
def baseline_numbers(table):
    nats = rank_baseline.heldout_baseline(
        table,
        BASELINE_MODELS,
        comparisons=rank_baseline.DEFAULT_COMPARISONS,
        n_starts=STRATEGY_STARTS,
        n_jobs=JOBS,
        cache_dir=CACHE_DIR / "baseline",
        shared_cache_dir=SHARED_STRATEGY_CACHE,
    )
    source = switch.analyze_switch_source(table, None, n_jobs=JOBS, cache_dir=CACHE_DIR / "switch")
    fits = {
        k: null_agents.fit_null_agents(table, k, null_agents.DEFAULT_DECAY)
        for k in switch.NULL_AGENTS
    }
    calibration = switch.calibrate_source(
        table,
        source,
        fits,
        replicates=SWITCH_REPLICATES,
        seed=SWITCH_SEED,
        n_jobs=JOBS,
        cache_dir=CACHE_DIR / "switch",
    )
    return nats, source, calibration


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    base_before = baseline_numbers(trials)
    base_after = baseline_numbers(selected)
strategy_after = base_after[0]

In [ ]:
def nats_summary(nats):
    wide = nats.nats.pivot(index="mouse", columns="model", values="nats_per_choice")[
        list(BASELINE_MODELS)
    ]
    return wide.mean().rename("mean nats per trial"), nats.differences.drop(columns="mouse").mean()


def switch_summary(source, calibration):
    raw = switch.source_summaries({"Mice": source})
    raw = raw[raw["term"] == "beta"].set_index(["condition", "value"])[
        ["n", "mean", "se", "p_wilcoxon"]
    ]
    z = calibration.summary
    z = z[z["term"] == "beta"].set_index(["agent", "condition", "value"])[
        ["n", "mean", "se", "p_wilcoxon"]
    ]
    return raw, z


for label, (nats, source, calibration) in (
    ("before the selection", base_before),
    ("after the selection", base_after),
):
    print("=" * 20, label, "-", nats.nats["mouse"].nunique(), "mice in the strategy comparison")
    means_, differences = nats_summary(nats)
    display(means_.round(4))
    display(
        differences.round(4).rename(
            "mean paired difference (first minus second, negative: first better)"
        )
    )
    raw, z = switch_summary(source, calibration)
    print("switch regression, raw value weight across mice (", len(source.decays), "mice )")
    display(raw.round(4))
    print("calibrated z of the value weight against each null agent")
    display(z.round(3))

**Reading the baseline.** The selection removes 10,351 of 49,572 trials (the problems 8 and later of MY_71_L, MY_71_R and MY_72_L) and changes nothing that matters: the held-out nats per trial move by 0.002 to 0.003 (WSLS (Magnitude) 0.7475 to 0.7449, WSLS (Habit) 0.6684 to 0.6661, Q-learner 0.7554 to 0.7541, kernel Q-learner 0.6628 to 0.6614) and every paired difference by less than 0.002 (the kernel Q-learner and WSLS (Habit) remain far ahead of WSLS (Magnitude) and the Q-learner, and WSLS (Habit) is 0.0056 nats per trial worse than the kernel Q-learner before and 0.0046 after). The switch regression's believed-value weight in the after-nonzero condition is 0.087 before and 0.094 after (Wilcoxon p 0.0009 both), its calibrated z against WSLS (Habit) 2.30 before and 2.28 after (p 0.001); the true value weight after a zero payout stays at about 0 (z 0.24 and 0.35, not significant). So the earlier finding (mice's switching follows what they believe each option is worth, more than any of the four null agents) is unchanged by the selection.

## 4. The rank tests on the mice

The kernel Q-learner fits of section 3 (the held-out comparison's full-data fits, on the selected table)
are the comparator parameters, together with the WSLS (Magnitude) and WSLS (Habit) fits.
`pipe.analyze_mice` runs the five tests on every mouse; `pipe.replay_source` plays each mouse's three
fitted agents on its own problems `REPLICATES` times; `pipe.calibrate_source` gives each mouse readout's
z and p against each agent. Mice with fewer than 200 stay/leave decisions or too few problems are skipped
with a warning and named.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fits = strategy_after.fits
    kq_fits = {m: f[strategy_models.KERNEL_Q] for m, f in fits.items()}
    mice = pipe.analyze_mice(selected, kq_fits, n_jobs=JOBS, cache_dir=CACHE_DIR / "mice")
print("skipped:", mice.skipped or "none")
rules = nulls.task_rules_by_cohort(selected)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    replays = pipe.replay_source(
        selected,
        fits,
        replicates=REPLICATES,
        rules=rules,
        seed=SEED,
        n_jobs=JOBS,
        cache_dir=CACHE_DIR / "replays",
    )
print("replays left out for too little data:")
display(replays.failed[replays.failed["n_failed"] > 0])
calibration = pipe.calibrate_source(mice.readouts, replays)
data = rank_plots.FigureData(mice, replays, calibration)

In [ ]:
rank_plots.plot_all(data, OUTPUT_DIR)
for name in rank_plots.FIGURE_NAMES:
    show(OUTPUT_DIR / name)

In [ ]:
pd.set_option("display.max_rows", None, "display.width", 220, "display.max_columns", None)
tested = [r for r in mice.readouts.columns if r != "mouse" and signatures.which_test(r)]
print("every mouse's readouts")
display(mice.readouts.set_index("mouse")[tested].round(3))
summary = calibration.summary
probe = summary["readout"].str.startswith("ws_")  # the worst-alternative probe
summary = summary[summary["readout"].isin(signatures.PREDICTED) | probe].copy()
summary["agent"] = summary["agent"].map(rank_plots.SOURCE_LABELS)
print("calibrated z across mice and who passes (p < 0.05 in the predicted direction)")
display(summary.round(3))
print("every mouse: z and p against the kernel Q-learner (the key comparator)")
key = calibration.table[calibration.table["agent"] == pipe.KEY_AGENT]
shown = key[key["readout"].isin(signatures.PREDICTED) | key["readout"].str.startswith("ws_")]
display(shown.pivot(index="mouse", columns="readout", values="z").round(2))

**Reading the figures: Test 1, stay depends on the second best, not the third.** Left: the weight of the best alternative's believed value in the stay model (negative: a better alternative lowers staying). Middle: the weight of the worst alternative's value. Right: worst minus best. Rank predicts `w_best` clearly negative and `w_worst` about 0; a softmax over values predicts both negative, the worst less. Mice: `w_best` -0.023 (se 0.010), the same as the kernel Q-learner's -0.021 (z -0.15, Wilcoxon p 0.72, 1 of 14 mice passes), so the best alternative does not lower staying more than the softmax learner predicts. `w_worst` is +0.176 (se 0.049), clearly positive, against +0.027 for the kernel Q-learner (z +1.46, p 0.004, 5 mice pass) and about -0.01 for the WSLS agents: a better believed worst alternative makes the mice stay *more*. `w_diff` (+0.199, z +1.19, p 0.003) is positive only because of this positive worst weight, not because the best alternative matters more. This is neither prediction: **not rank-like as predicted; an unexpected positive worst-alternative weight.** The next section probes where it comes from.

**Reading the figures: probe of the positive worst-alternative weight.** The task has exactly one zero arm at all times, so after an own payout above 0 one alternative must be the zero arm: two alternatives believed to pay means a stale belief (an arm turned zero since it was last visited), which might make leaving less attractive without any comparison of values. Top row: the worst weight of test 1 (+0.176), with the trials since each alternative was last visited added (+0.180: ages change nothing), with a stale-belief flag (the worst alternative is believed to pay) added (+0.113, se 0.052; the kernel Q-learner goes to -0.048, so z against it falls from +1.46 to +0.94 but stays significant, p 0.009), and the flag's own weight (+0.175, se 0.102; the kernel Q-learner's replays give +0.169, z +0.12). So the flag explains about a third of the gap, and its effect is a feature of the schedule that the fitted learner reproduces, not something specific to the mice. Bottom row: within own payouts above 0 the worst weight is +0.140 (kernel Q-learner -0.090, z +1.32, p 0.0006) and the flag has no effect there (-0.045); within own payouts of 0 the worst weight is +0.022 (z +0.35, not significant). P(stay) after a payout above 0 is 0.774 when the beliefs are consistent with one zero arm and 0.781 when stale (kernel Q-learner 0.763 and 0.760): stale beliefs do not change staying after a payout, 15% of those decisions. After a zero payout stale beliefs raise staying (0.505 against 0.363, kernel Q-learner 0.617 against 0.463), 38% of those decisions, an effect the agents show as well. Artifact checks: the unpenalized refit gives +0.197 (ridge +0.180); the worst value's variance inflation is 1.6 (5.0 once the flag it defines is in the model, largest of any regressor 8.7, below 10); its correlation with the own payout is -0.40 and with the best alternative +0.34, the same as in the kernel Q-learner's replays (-0.40, +0.36). The weight is not a ridge or a collinearity artifact. **Verdict: the one-zero-arm structure explains part of the weight (about a third, through stale beliefs, in a way the softmax learner shares) and the rest, +0.11 after the flag and +0.14 within paid decisions, is not explained by it: the mice stay more when the worst alternative is believed to pay more, an effect neither account predicts and that the three fitted agents do not produce.**

**Reading the figures: Test 2, abrupt switching at believed crossings.** Left: P(leave) by `D`, the own arm's believed value minus the best alternative's. A rank algorithm drops at `D` = 0 as a step and is flat on both sides. Middle: the step weight at `D` < 0 (positive: more leaving when the alternative is believed better, over and above the linear value). Right: the nats gained by a step over a smooth curve. Mice: the curve is graded, falling smoothly from 0.55 at `D` = -3 to 0.17 at `D` = 0 and rising again slightly; the step weight is 0.161 (se 0.088), the same as the kernel Q-learner's 0.177 (z -0.08, p 1.0, no mouse passes); WSLS agents lie below the mice (-0.10, z +1.5, p 0.003) but they cannot see believed values. The gain of the step over a smooth curve is -0.007 nats (a step does not fit better) and z -0.28 against the Q-learner. Aligned to believed crossings (figure Rank Crossing Aligned, 759 events): on the first trial after a crossing the mice leave in 0.34 of cases (kernel Q-learner 0.27, WSLS (Magnitude) 0.43) and adopt the new believed best in 0.13 (0.14 for the kernel Q-learner); after 10 trials the median mouse still has not adopted it (`cross_t50` = 11 for every mouse, so there is no z). The mice are not abrupt: they match the kernel Q-learner, and `cross_leave_1` is 0.82 z above it only because the replays leave less often (1 of 14 mice passes).

**Reading the figures: Test 3, the third option is categorical.** Left: P(choose the arm believed worst) by its gap to the second best (pulses). A rank algorithm is flat (the third is never compared with anything); a softmax learner falls with the gap. Middle: the worst-arm slope over the gap (log odds per pulse). Right: the worst arm's weight in the choice between the top two. Mice: slope -0.065 (se 0.035); WSLS (Magnitude) -0.117, WSLS (Habit) -0.103, kernel Q-learner -0.149. The mice's slope is the flattest, in the rank direction (z +0.70 against the kernel Q-learner, Wilcoxon p 0.078, 11 of 14 positive, 1 mouse passes), but it is not zero (P(worst) still falls only from 0.16 to 0.15 with the gap in the mean curve) and it is not significantly different from the softmax comparator. The top-two panel is a check, not a pass criterion: in the choice between the top two the worst arm's weight is scattered around 0 (mean -0.18, se 0.15; five mice beyond 0.5 in either direction). The learners' replays also have a small slope, so the test has some power only for a learner as sharp as the toy.

**Reading the figures: Test 4, ranks matter more than values.** Left: held-out nats per trial of the rank model over the value model in the stay model (positive: ranks predict better); middle: the same in the switch model; right: the scale slope, how much the stay probability at the best-ranked own arm changes with the payout's magnitude (rank: 0, softmax: positive; smaller is rank-like). Held-out nats: the stay model prefers values slightly (mice -0.006, kernel Q-learner -0.007, z +0.20, p 0.54) and the switch model is a tie (mice +0.003, kernel Q-learner -0.002, z +1.03, p 0.19, 5 mice pass); the difference is of the same size as the agents' and these are not significant. Scale: P(stay) at the best-ranked own arm rises 0.146 per magnitude step (se 0.023) in the mice, against 0.385 for the kernel Q-learner (z -2.51, p < 0.001, 10 of 14 mice pass), 0.277 for WSLS (Magnitude) (z -1.58, p 0.001) and 0.223 for WSLS (Habit) (z -0.89, p 0.005). The mice stay at a rank less dependent on magnitude than every agent fitted to them, but not independent of it (0.146 is not 0, and the toy rank agent is 0.07).

**Reading the figures: Test 5, transfer across problems and inference at block changes.** Transfer: P(best) in the first 100 trials and the trials to threshold per problem number (left two panels), and their slopes across problems (right two). A rank algorithm transfers: P(best) rises and the threshold time falls. Mice: the slope of P(best) is +0.010 per problem (se 0.012; kernel Q-learner +0.005, z +0.04, p 0.95) and the slope of the trials to threshold -3.3 (se 2.1; z -0.18, p 0.34); the mean curve of P(best) is flat at 0.45 to 0.5 and falls in problem 10 (one mouse). One mouse has a clear positive slope (MY_66_LR, +0.12, passes; MY_66_LL has only 2 problems and no slope). Block inference: after a block change the first leave goes to the old zero arm in 0.56 of cases (se 0.025; 739 events from 14 mice), against 0.51 for the kernel Q-learner (z +0.56, p 0.24, 2 mice pass) and 0.48 and 0.50 for the WSLS agents; where the believed values say the third arm is better it is 0.55. The mice are at most slightly above the 0.5 that choosing at random between the two other arms gives, so no sign of inferring the new best without sampling.

**Verdict, against the kernel Q-learner fitted to each mouse (n = 14 mice; ranks are not tested on a mouse count below 6).**

| Test | Mice | Kernel Q-learner | z, p | Mice passing | Verdict |
|---|---|---|---|---|---|
| 1 `w_best` | -0.023 | -0.021 | -0.15, 0.72 | 1 of 14 | softmax-like (no stronger than the learner) |
| 1 `w_worst` | +0.176 | +0.027 | +1.46, 0.004 | 5 of 14 | unexpected positive weight; not rank-like as predicted |
| 2 `step_weight` | +0.161 | +0.177 | -0.08, 1.0 | 0 of 14 | softmax-like |
| 3 `worst_slope` | -0.065 | -0.149 | +0.70, 0.078 | 1 of 14 | not distinguishable (leans rank) |
| 4 `scale_slope_first` | 0.146 | 0.385 | -2.51, < 0.001 | 10 of 14 | rank-like on scale; held-out nats tie |
| 5 `slope_p100`, `block_inherit` | +0.010, 0.560 | +0.005, 0.514 | +0.04 (0.95), +0.56 (0.24) | 1 and 2 of 14 | softmax-like / not testable |

The mice are not a clean copy of either account. Only test 4's scale slope goes clearly in the rank direction; the best alternative's weight, the crossing and the transfer match the softmax learner, the third option cannot be separated from it, and the mice's worst-alternative weight is positive, which neither account predicts (the one-zero-arm structure explains about a third of it). `cross_t50` (censored at 11 for every mouse) and `slope_p100` for MY_66_LL (2 problems) cannot be tested.